Title: check_CESM2_files.ipynb

Purpose: Look at CESM2_LE files

Author: Onno Nennecke on 12.02.2026 Modified: 16.02.2026

Input data: 

- CESM2_LE files
    - This file lies here: /climca/data/CESM2_LE

Output data:

- List of all usable runs
    - This file lies here: 


In [1]:
# Load necessary libraries
import xarray as xr
import os
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display


In [2]:
middle_year = 2044
start_year = middle_year - 5
end_year = middle_year + 4

print(f"Looking for files covering {start_year}-{end_year}...")

Looking for files covering 2039-2048...


In [3]:
# Variables
vars = ["U10", "FSDS", "PSL", "TREFHTMX", "TREFHT"]

In [4]:
# Create a DataFrame to store results
results = pd.DataFrame(columns=vars)
results

,U10,FSDS,PSL,TREFHTMX,TREFHT


In [5]:
for var in vars:
    print(f"\nChecking variable: {var}")

    dir = Path(f'/climca/data/CESM2_LE/{var}/day_raw/')
    
    files = list(dir.glob("*.nc"))

    records = []

    for f in files:
        parts = f.name.split(".")

        # Defensive check in case some filenames differ
        if len(parts) < 11:
            continue

        record = {
            "run_type": parts[0],
            "model_version": parts[1],
            "experiment": parts[2],
            "resolution": parts[3],
            "LE_ID_frcng": parts[4],
            "ensemble_member": parts[5],  # e.g. 009
            'run_id': parts[4] + '_' + parts[5],  # 
            "component": parts[6],         # cam
            "frequency": parts[7],         # h6
            "variable": parts[8],          # U
            "timerange": parts[9],
            "filename": f.name
        }
    # 'b.e21.BSSP370smbb.f09_g17.LE2-1011.001.cam.h1.FSDS.20650101-20741231.nc'
        records.append(record)

    # Put into a DataFrame (very handy)
    df = pd.DataFrame(records)
    timeranges = ["20150101-20241231", "20350101-20441231", "20450101-20541231"]
    df = df[df["timerange"].isin(timeranges)]
    # print("\nPreview:")
    # print(df.head())
    print(f"Total files found relevant: {len(df)}")

    df.sort_values(by=['LE_ID_frcng', 'ensemble_member', 'timerange'], inplace=True)
    # df.sort_values(by=['run_id'], inplace=True)

    results[var] = df["run_id"].unique()

    # ---------------------------
    # Counts & summaries
    # ---------------------------

    # print("\nFiles per experiment member:")
    # print(df["experiment"].value_counts().sort_index())

    # print("\nFiles per ensemble member:")
    # print(df["ensemble_member"].value_counts())

    print("\nFiles per timerange:")
    print(df["timerange"].value_counts())



Checking variable: U10
Total files found relevant: 300

Files per timerange:
timerange
20150101-20241231    100
20350101-20441231    100
20450101-20541231    100
Name: count, dtype: int64

Checking variable: FSDS
Total files found relevant: 300

Files per timerange:
timerange
20150101-20241231    100
20350101-20441231    100
20450101-20541231    100
Name: count, dtype: int64

Checking variable: PSL
Total files found relevant: 300

Files per timerange:
timerange
20150101-20241231    100
20350101-20441231    100
20450101-20541231    100
Name: count, dtype: int64

Checking variable: TREFHTMX
Total files found relevant: 300

Files per timerange:
timerange
20150101-20241231    100
20350101-20441231    100
20450101-20541231    100
Name: count, dtype: int64

Checking variable: TREFHT
Total files found relevant: 300

Files per timerange:
timerange
20150101-20241231    100
20350101-20441231    100
20450101-20541231    100
Name: count, dtype: int64


In [6]:
# Remove variable and filename columns from df and save as CSV
df.drop(columns=["variable", "filename", 'timerange'], inplace=True)
# Remove duplicate rows if they exist
df = df.loc[~df.duplicated()]

df.to_csv(f'/home/onennecke/CMIP_models/CESM2_LE_runs_long.csv', index=False)

In [15]:
# Create a new DataFrame
new_df = pd.DataFrame(columns=['ESM', 'Institution', 'run'])
new_df['run'] = df['run_id']
new_df['ESM'] = 'CESM2'
new_df['Institution'] = 'NCAR'

new_df.to_csv(f'/home/onennecke/CMIP_models/CESM2_LE_runs.csv', index=False)

In [8]:
# Check if all columns are the same
results['Check'] = results.apply(lambda row: len(set(row[vars])) == 1, axis=1)
results
np.sum(results['Check'])

np.int64(100)

In [18]:
df

display(df.style.set_sticky())

,run_type,model_version,experiment,resolution,LE_ID_frcng,ensemble_member,run_id,component,frequency
986,b,e21,BSSP370cmip6,f09_g17,LE2-1001,001,LE2-1001_001,cam,h1
1608,b,e21,BSSP370smbb,f09_g17,LE2-1011,001,LE2-1011_001,cam,h1
1784,b,e21,BSSP370cmip6,f09_g17,LE2-1021,002,LE2-1021_002,cam,h1
1715,b,e21,BSSP370smbb,f09_g17,LE2-1031,002,LE2-1031_002,cam,h1
2313,b,e21,BSSP370cmip6,f09_g17,LE2-1041,003,LE2-1041_003,cam,h1
1063,b,e21,BSSP370smbb,f09_g17,LE2-1051,003,LE2-1051_003,cam,h1
957,b,e21,BSSP370cmip6,f09_g17,LE2-1061,004,LE2-1061_004,cam,h1
282,b,e21,BSSP370smbb,f09_g17,LE2-1071,004,LE2-1071_004,cam,h1
1904,b,e21,BSSP370cmip6,f09_g17,LE2-1081,005,LE2-1081_005,cam,h1
604,b,e21,BSSP370smbb,f09_g17,LE2-1091,005,LE2-1091_005,cam,h1


In [17]:
new_df

,ESM,Institution,run
986,CESM2,NCAR,LE2-1001_001
1608,CESM2,NCAR,LE2-1011_001
1784,CESM2,NCAR,LE2-1021_002
1715,CESM2,NCAR,LE2-1031_002
2313,CESM2,NCAR,LE2-1041_003
...,...,...,...
399,CESM2,NCAR,LE2-1301_016
404,CESM2,NCAR,LE2-1301_017
2361,CESM2,NCAR,LE2-1301_018
2120,CESM2,NCAR,LE2-1301_019


In [ ]:
# Read one test file to check the time coverage

ds = xr.open_dataset(
    "/climca/data/CESM2_LE/U10/day_raw/b.e21.BSSP370smbb.f09_g17.LE2-1011.001.cam.h1.U10.20450101-20541231.nc",
    engine="netcdf4"
)
ds.load()

<xarray.Dataset> Size: 808MB
Dimensions:              (lat: 192, lon: 288, lev: 32, ilev: 33, cosp_prs: 7,
                          nbnd: 2, cosp_tau: 7, cosp_scol: 10, cosp_ht: 40,
                          cosp_sr: 15, cosp_sza: 5, cosp_dbze: 15,
                          cosp_htmisr: 16, cosp_tau_modis: 7, cosp_reffice: 6,
                          cosp_reffliq: 6, time: 3650)
Coordinates: (12/16)
  * lat                  (lat) float64 2kB -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                  (lon) float64 2kB 0.0 1.25 2.5 ... 356.2 357.5 358.8
  * lev                  (lev) float64 256B 3.643 7.595 14.36 ... 976.3 992.6
  * ilev                 (ilev) float64 264B 2.255 5.032 10.16 ... 985.1 1e+03
  * cosp_prs             (cosp_prs) float64 56B 9e+04 7.4e+04 ... 2.45e+04 9e+03
  * cosp_tau             (cosp_tau) float64 56B 0.15 0.8 2.45 ... 41.5 100.0
    ...                   ...
  * cosp_dbze            (cosp_dbze) float64 120B -72.5 -42.5 ... 17.5 50.0
  * cosp_htmisr          (cosp_htmisr) float64 128B 0.0 250.0 ... 1.8e+04
  * cosp_tau_modis       (cosp_tau_modis) float64 56B 0.15 0.8 ... 41.5 100.0
  * cosp_reffice         (cosp_reffice) float64 48B 5e-06 1.5e-05 ... 7.5e-05
  * cosp_reffliq         (cosp_reffliq) float64 48B 4e-06 9e-06 ... 2.5e-05
  * time                 (time) object 29kB 2045-01-01 00:00:00 ... 2054-12-3...
Dimensions without coordinates: nbnd
Data variables: (12/35)
    gw                   (lat) float64 2kB ...
    hyam                 (lev) float64 256B ...
    hybm                 (lev) float64 256B ...
    P0                   float64 8B ...
    hyai                 (ilev) float64 264B ...
    hybi                 (ilev) float64 264B ...
    ...                   ...
    n2ovmr               (time) float64 29kB ...
    f11vmr               (time) float64 29kB ...
    f12vmr               (time) float64 29kB ...
    sol_tsi              (time) float64 29kB ...
    nsteph               (time) int32 15kB ...
    U10                  (time, lat, lon) float32 807MB ...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BSSP370smbb.f09_g17.LE2-1011.001
    logname:           sunseon
    host:              mom1
    initial_file:      b.e21.BHISTsmbb.f09_g17.LE2-1011.001.cam.i.2015-01-01-...
    topography_file:   /mnt/lustre/share/CESM/cesm_input/atm/cam/topo/fv_0.9x...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1